# 1. Carga

In [1]:
import geopandas as gpd

path_mun    = "datos/territorio_municipios/recintos_municipales_inspire_peninbal_etrs89/recintos_municipales_inspire_peninbal_etrs89.shp"
path_prov   = "datos/territorio_municipios/recintos_provinciales_inspire_peninbal_etrs89/recintos_provinciales_inspire_peninbal_etrs89.shp"
path_clm    = "datos/territorio_municipios/recintos_autonomicas_inspire_peninbal_etrs89/recintos_autonomicas_inspire_peninbal_etrs89.shp"

gdf_mun     = gpd.read_file(path_mun)
gdf_prov    = gpd.read_file(path_prov)
gdf_clm    = gpd.read_file(path_clm)

In [2]:
gdf_mun.crs

<Geographic 2D CRS: EPSG:4258>
Name: ETRS89
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: Europe - onshore and offshore: Albania; Andorra; Austria; Belgium; Bosnia and Herzegovina; Bulgaria; Croatia; Czechia; Denmark; Estonia; Faroe Islands; Finland; France; Germany; Gibraltar; Greece; Hungary; Ireland; Italy; Kosovo; Latvia; Liechtenstein; Lithuania; Luxembourg; Malta; Moldova; Monaco; Montenegro; Netherlands; North Macedonia; Norway including Svalbard and Jan Mayen; Poland; Portugal - mainland; Romania; San Marino; Serbia; Slovakia; Slovenia; Spain - mainland and Balearic islands; Sweden; Switzerland; United Kingdom (UK) including Channel Islands and Isle of Man; Vatican City State.
- bounds: (-16.1, 33.26, 38.01, 84.73)
Datum: European Terrestrial Reference System 1989 ensemble
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [3]:
gdf_mun.columns

Index(['INSPIREID', 'COUNTRY', 'NATLEV', 'NATLEVNAME', 'NATCODE', 'NAMEUNIT',
       'CODNUT1', 'CODNUT2', 'CODNUT3', 'geometry'],
      dtype='str')

In [4]:
gdf_mun.head()

,INSPIREID,COUNTRY,NATLEV,NATLEVNAME,NATCODE,NAMEUNIT,CODNUT1,CODNUT2,CODNUT3,geometry
0,ES.IGN.BDDAE.34100303075,ES,https://inspire.ec.europa.eu/codelist/Administ...,Municipio,34100303075,el Castell de Guadalest,ES5,ES52,ES521,"POLYGON ((-0.16635 38.64417, -0.16663 38.6439,..."
1,ES.IGN.BDDAE.34100303076,ES,https://inspire.ec.europa.eu/codelist/Administ...,Municipio,34100303076,Guardamar del Segura,ES5,ES52,ES521,"POLYGON ((-0.69926 38.11493, -0.65415 38.1093,..."
2,ES.IGN.BDDAE.34010404001,ES,https://inspire.ec.europa.eu/codelist/Administ...,Municipio,34010404001,Abla,ES6,ES61,ES611,"POLYGON ((-2.78452 37.0935, -2.7841 37.09476, ..."
3,ES.IGN.BDDAE.34010404002,ES,https://inspire.ec.europa.eu/codelist/Administ...,Municipio,34010404002,Abrucena,ES6,ES61,ES611,"POLYGON ((-2.88868 37.09169, -2.8886 37.09206,..."
4,ES.IGN.BDDAE.34010404003,ES,https://inspire.ec.europa.eu/codelist/Administ...,Municipio,34010404003,Adra,ES6,ES61,ES611,"POLYGON ((-3.14019 36.78779, -3.13982 36.78802..."


In [5]:
len(gdf_mun)

8132

# 2. Municipios

## 2.1. Observación de la codificación

Los últimos 5 dígitos del NATCODE parecen ser equivalentes al código de municipio, al menos en Albacete

In [6]:
gdf_mun[gdf_mun["NAMEUNIT"] == "Albacete"]

,INSPIREID,COUNTRY,NATLEV,NATLEVNAME,NATCODE,NAMEUNIT,CODNUT1,CODNUT2,CODNUT3,geometry
4274,ES.IGN.BDDAE.34080202003,ES,https://inspire.ec.europa.eu/codelist/Administ...,Municipio,34080202003,Albacete,ES4,ES42,ES421,"POLYGON ((-2.18035 39.1, -2.16738 39.10308, -2..."


Parece que los últimos 5 dígitos son únicos para cada municipio

In [7]:
gdf_mun["NATCODE"].sort_values().str[-5:].unique()

<ArrowStringArray>
['04001', '04002', '04003', '04004', '04005', '04006', '04007', '04008',
 '04009', '04010',
 ...
 '53056', '51001', '52001', '54001', '54002', '54003', '54004', '54005',
 '54006', '54007']
Length: 8132, dtype: str

Comprobando con Abengibre, coincide también con el código 02001

In [8]:
gdf_mun[gdf_mun["NAMEUNIT"] == "Abengibre"]["NATCODE"].str[-5:]

4272    02001
Name: NATCODE, dtype: str

## 2.2. Reducción de los datos a Castilla-La Mancha y columnas esenciales

In [9]:
PRE_CLM = ("02", "13", "16", "19", "45")
gdf_mun = gdf_mun.loc[gdf_mun["NATCODE"].str[-5:-3].isin(PRE_CLM), ["NATCODE", "NAMEUNIT", "geometry"]]
gdf_mun["NATCODE"] = gdf_mun["NATCODE"].str[-5:]
gdf_mun = gdf_mun.rename(columns={"NATCODE": "cod_mun", "NAMEUNIT": "nombre"})
gdf_mun.head()

,cod_mun,nombre,geometry
4272,02001,Abengibre,"POLYGON ((-1.58319 39.20444, -1.58315 39.20462..."
4273,02002,Alatoz,"POLYGON ((-1.4171 39.07751, -1.41202 39.07629,..."
4274,02003,Albacete,"POLYGON ((-2.18035 39.1, -2.16738 39.10308, -2..."
4275,02004,Albatana,"POLYGON ((-1.54628 38.5671, -1.54965 38.56982,..."
4276,02005,Alborea,"POLYGON ((-1.42629 39.24501, -1.42645 39.24632..."


## 2.3. Guardado

Se puede definir una tolerancia para la simplificación, que es aproximadamente igual a la raiz del área de triangulos eliminados.

In [10]:
# import shapely

# print(shapely.get_num_coordinates(gdf_mun.geometry).sum())
# gdf_m = gdf_mun.to_crs(epsg=25830) # A metros: https://epsg.io/25830
# # https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.simplify_coverage.html
# gdf_m["geometry"] = gdf_m.simplify_coverage(tolerance=30)
# # Vuelve a ETRS89
# gdf_m = gdf_m.to_crs(epsg=4258) # Vuelta a ETRS89: https://epsg.io/4258
# print(shapely.get_num_coordinates(gdf_m.geometry).sum())

In [11]:
gdf_mun.to_parquet("data/region_municipios_clm.parquet", index=False)

# 3. Provincia

## 3.1. Observación de la codificación

In [12]:
gdf_prov.head()

,INSPIREID,COUNTRY,NATLEV,NATLEVNAME,NATCODE,NAMEUNIT,CODNUT1,CODNUT2,CODNUT3,geometry
0,ES.IGN.BDDAE.34090800000,ES,https://inspire.ec.europa.eu/codelist/Administ...,Provincia,34090800000,Barcelona,ES5,ES51,None,"MULTIPOLYGON (((1.75125 41.21822, 1.75106 41.2..."
1,ES.IGN.BDDAE.34070900000,ES,https://inspire.ec.europa.eu/codelist/Administ...,Provincia,34070900000,Burgos,ES4,ES41,None,"MULTIPOLYGON (((-3.96817 41.58462, -3.96879 41..."
2,ES.IGN.BDDAE.34111000000,ES,https://inspire.ec.europa.eu/codelist/Administ...,Provincia,34111000000,Cáceres,ES4,ES43,None,"POLYGON ((-6.80634 39.39066, -6.8069 39.39081,..."
3,ES.IGN.BDDAE.34011100000,ES,https://inspire.ec.europa.eu/codelist/Administ...,Provincia,34011100000,Cádiz,ES6,ES61,None,"MULTIPOLYGON (((-6.00648 36.18153, -6.00633 36..."
4,ES.IGN.BDDAE.34101200000,ES,https://inspire.ec.europa.eu/codelist/Administ...,Provincia,34101200000,Castelló/Castellón,ES5,ES52,None,"MULTIPOLYGON (((-0.38276 39.79419, -0.38788 39..."


In [13]:
gdf_prov[gdf_prov["NAMEUNIT"] == "Albacete"]["NATCODE"].str[-7:]

46    0200000
Name: NATCODE, dtype: str

## 3.2. Reducción a Castilla-La Mancha y columnas esenciales

In [14]:
gdf_prov = gdf_prov.loc[gdf_prov["NATCODE"].str[-7:-5].isin(PRE_CLM), ["NATCODE", "NAMEUNIT", "geometry"]]
gdf_prov["NATCODE"] = gdf_prov["NATCODE"].str[-7:-5]
gdf_prov = gdf_prov.rename(columns={"NATCODE": "cod_prov", "NAMEUNIT": "nombre"})
gdf_prov.head()

,cod_prov,nombre,geometry
5,13,Ciudad Real,"MULTIPOLYGON (((-4.15142 38.38293, -4.15379 38..."
8,16,Cuenca,"POLYGON ((-2.66346 39.32265, -2.66449 39.32316..."
11,19,Guadalajara,"MULTIPOLYGON (((-3.18674 40.27899, -3.18588 40..."
35,45,Toledo,"POLYGON ((-4.74654 39.49317, -4.74707 39.49355..."
46,02,Albacete,"POLYGON ((-2.55212 38.08501, -2.55207 38.08517..."


## 3.3. Guardado

In [15]:
gdf_prov.to_parquet("data/region_provincias_clm.parquet", index=False)

## 4. Castilla-La Mancha

In [16]:
gdf_clm.head()

,INSPIREID,COUNTRY,NATLEV,NATLEVNAME,NATCODE,NAMEUNIT,CODNUT1,CODNUT2,CODNUT3,geometry
0,ES.IGN.BDDAE.34010000000,ES,https://inspire.ec.europa.eu/codelist/Administ...,Comunidad autónoma,34010000000,Andalucía,ES6,ES61,None,"MULTIPOLYGON (((-6.30303 36.54817, -6.30305 36..."
1,ES.IGN.BDDAE.34020000000,ES,https://inspire.ec.europa.eu/codelist/Administ...,Comunidad autónoma,34020000000,Aragón,ES2,ES24,None,"POLYGON ((-1.68752 40.57939, -1.68582 40.57991..."
2,ES.IGN.BDDAE.34030000000,ES,https://inspire.ec.europa.eu/codelist/Administ...,Comunidad autónoma,34030000000,Principado de Asturias,ES1,ES12,None,"MULTIPOLYGON (((-6.85727 43.17325, -6.85767 43..."
3,ES.IGN.BDDAE.34040000000,ES,https://inspire.ec.europa.eu/codelist/Administ...,Comunidad autónoma,34040000000,Illes Balears,ES5,ES53,None,"MULTIPOLYGON (((1.24355 38.86118, 1.2436 38.86..."
4,ES.IGN.BDDAE.34060000000,ES,https://inspire.ec.europa.eu/codelist/Administ...,Comunidad autónoma,34060000000,Cantabria,ES1,ES13,None,"MULTIPOLYGON (((-4.15851 42.86621, -4.16084 42..."


In [17]:
gdf_clm = gdf_clm.loc[gdf_clm["NAMEUNIT"] == "Castilla-La Mancha", ["NAMEUNIT", "geometry"]]
gdf_clm = gdf_clm.rename(columns={"NAMEUNIT": "nombre"})
gdf_clm.head()

,nombre,geometry
6,Castilla-La Mancha,"MULTIPOLYGON (((-4.49534 38.44132, -4.49591 38..."


In [18]:
gdf_clm.to_parquet("data/region_clm.parquet", index=False)